# Lab 4 — Proximity Detector (LOF)

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

> **Quick check:** precision ≈ **0.33** · recall (fraud) = **1.00** · train legit **792** rows




## Semi-supervised anomaly detection

Train on **legitimate** transactions only; label **-1** = outlier.

In [ ]:
from pathlib import Path

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES = ["amount", "distance_from_home"]
CATEGORICAL_FEATURES = ["merchant_category"]

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_legit = X_train[y_train == 0]
print(f"train total: {len(X_train)} (fraud {int(y_train.sum())})")
print(f"train legit only: {len(X_train_legit)}")
print(f"test: {len(X_test)} (fraud {int(y_test.sum())})")


## Preprocess and fit LOF on legit train only

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

X_train_legit_s = preprocess.fit_transform(X_train_legit)
X_test_s = preprocess.transform(X_test)

CONTAMINATION = 0.02
lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION, novelty=True)
lof.fit(X_train_legit_s)

pred = lof.predict(X_test_s)
y_pred = np.where(pred == -1, 1, 0)

print("Lab 4 — Proximity detector (LOF)")
print(f"train legit rows: {len(X_train_legit)}")
print(f"test rows: {len(X_test)}")
print(f"predicted anomalies: {int((y_pred == 1).sum())}")


## Precision and recall (fraud)

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"precision (fraud): {precision:.4f}")
print(f"recall (fraud): {recall:.4f}")

results = pd.DataFrame({
    "metric": ["precision (fraud)", "recall (fraud)"],
    "value": [precision, recall],
})
display(results.round(4))


## Prediction breakdown

In [ ]:
breakdown = pd.DataFrame({
    "actual_fraud": y_test.values,
    "predicted_fraud": y_pred,
    "amount": X_test["amount"].values,
    "distance_from_home": X_test["distance_from_home"].values,
})
display(breakdown.sort_values("predicted_fraud", ascending=False).head(10).round(2))


## Extension — vary contamination

In [ ]:
rows = []
for contam in [0.01, 0.02, 0.05]:
    model = LocalOutlierFactor(n_neighbors=20, contamination=contam, novelty=True)
    model.fit(X_train_legit_s)
    yp = np.where(model.predict(X_test_s) == -1, 1, 0)
    rows.append({
        "contamination": contam,
        "predicted_anomalies": int((yp == 1).sum()),
        "precision": precision_score(y_test, yp, zero_division=0),
        "recall": recall_score(y_test, yp, zero_division=0),
    })

display(pd.DataFrame(rows).round(4))


### LOF prompt 1

Show LOF raw predict labels.

In [ ]:
print(np.unique(pred, return_counts=True))

### LOF prompt 2

Count true positives.

In [ ]:
print(int(((y_pred==1)&(y_test==1)).sum()))

### LOF prompt 3

Count false positives.

In [ ]:
print(int(((y_pred==1)&(y_test==0)).sum()))

### LOF prompt 4

Count false negatives.

In [ ]:
print(int(((y_pred==0)&(y_test==1)).sum()))

### LOF prompt 5

Display contamination used.

In [ ]:
print(CONTAMINATION)

### LOF prompt 6

Show n_neighbors setting.

In [ ]:
print(lof.n_neighbors)

### LOF prompt 7

List predicted anomaly amounts.

In [ ]:
print(breakdown.loc[breakdown['predicted_fraud']==1, 'amount'].round(2).tolist())

### LOF prompt 8

Compare train legit vs total train.

In [ ]:
print(len(X_train_legit), len(X_train))

### LOF prompt 9

Verify novelty mode.

In [ ]:
print(lof.novelty)

### LOF prompt 10

Show test fraud rows in breakdown.

In [ ]:
display(breakdown.loc[breakdown['actual_fraud']==1].round(2))

### LOF prompt 11

Compute anomaly rate on test.

In [ ]:
print(round((y_pred==1).mean(), 4))

### LOF prompt 12

Re-run precision manually.

In [ ]:
print(precision_score(y_test, y_pred, zero_division=0))

### LOF prompt 13

Re-run recall manually.

In [ ]:
print(recall_score(y_test, y_pred, zero_division=0))

### LOF prompt 14

Summarize LOF mapping.

In [ ]:
print('LOF -1 mapped to fraud prediction 1')

### LOF prompt 15

Show feature matrix shapes.

In [ ]:
print(X_train_legit_s.shape, X_test_s.shape)

### LOF prompt 16

State semi-supervised assumption.

In [ ]:
print('Train density on legit-only; score all test rows.')

### LOF prompt 17

Compare to Day 5 DBSCAN noise idea.

In [ ]:
print('Both flag sparse/outlier points in feature space.')

### LOF prompt 18

Display top distances among flagged.

In [ ]:
display(breakdown.loc[breakdown['predicted_fraud']==1].nlargest(3, 'distance_from_home'))

### LOF prompt 19

Reconfirm test size.

In [ ]:
print(len(X_test))

### LOF prompt 20

Bridge to Random Forest.

In [ ]:
print('Lab 5 uses supervised RF with class_weight balanced.')

### Lab 4 quick recap 1

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 1: completed")

### Lab 4 quick recap 2

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 2: completed")

### Lab 4 quick recap 3

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 3: completed")

### Lab 4 quick recap 4

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 4: completed")

### Lab 4 quick recap 5

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 5: completed")

## Final checkpoint

In [ ]:
assert len(X_train_legit) == 792
assert len(X_test) == 200
assert int((y_pred == 1).sum()) == 6
assert abs(precision - 0.3333) < 0.05
assert abs(recall - 1.0) < 0.01
print("Numbers match — you're good.")



## Reflection

When is high recall at the cost of precision acceptable?